In [1]:
import os, urllib.request

uniref_path = 'data/uniref50.fasta'
if not os.path.exists(uniref_path):
    print('Downloading 10,000 UniRef50 sequences from UniProt...')
    url = 'https://rest.uniprot.org/uniref/stream?query=identity:0.5&format=fasta'
    req = urllib.request.Request(url)
    count = 0
    with open(uniref_path, 'w') as f, urllib.request.urlopen(req) as response:
        for line in response:
            line_str = line.decode('utf-8')
            if line_str.startswith('>'):
                count += 1
                if count > 10000:
                    break
            f.write(line_str)
    print(f'Successfully downloaded 10,000 sequences to {uniref_path}')
else:
    print(f'{uniref_path} already exists.')


Successfully downloaded 10,000 sequences to data/uniref50.fasta


In [2]:
!pip install evodiff
# EvoDiff dependencies (might require specific CUDA versions)
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 17.6 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 101.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 24.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.6/886.6 KB 101.7 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd
from Bio import SeqIO

msa_path = 'data/valid_msas.csv'
if not os.path.exists(msa_path):
    print('Creating a placeholder valid_msas.csv for OpenFold...')
    # In a real scenario, this requires downloading OpenFold MSAs (hundreds of GBs).
    # Here we create a mock file so the pipeline runs, using some controls as placeholders.
    mock_data = {'id': ['mock1', 'mock2'], 'sequence': ['MKTLLILAVIMSFGATAA', 'MLARALLPRATARLLQ']}
    pd.DataFrame(mock_data).to_csv(msa_path, index=False)
    print(f'Saved mock MSAs to {msa_path}')


Creating a placeholder valid_msas.csv for OpenFold...
Saved mock MSAs to data/valid_msas.csv


# Notebook 09 — EvoDiff Generalisation & FPR Datasets

**Block 5 from next_steps.md**
- Dataset 1: UniRef50 (Negative / Control Sequences) FPR
- Dataset 2: OpenFold MSAs (Structure-Conditioned Hard Negatives) FPR
- Dataset 3: EvoDiff Redesigns (Sequence-space and MSA-conditioned diffusion)
- Dataset 4: IDR Sequences (Structureless Negatives)

This notebook tests the probe on entirely new datasets and redesign methods to validate out-of-distribution robustness.

In [4]:
import warnings, json, random, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import EsmForMaskedLM, AutoTokenizer
from Bio import SeqIO
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_LAYER = 33
Path('results/evodiff').mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

# Load probe
class ToxinProbe(nn.Module):
    def __init__(self, d=1280):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    def forward(self, x):
        return self.linear(x).squeeze(-1)

# Note: The probe should ideally be reloaded from a saved state dict, 
# but we can recreate it by loading the saved weights direction.
probe = ToxinProbe().to(DEVICE)
try:
    # If you saved probe weights:
    probe.linear.weight.data = torch.tensor(np.load('results/probe_direction.npy')).unsqueeze(0).to(DEVICE)
    # Note: bias would be needed for exact probabilities, assume bias=0 for now or load if available.
except:
    print('Warning: Ensure probe weights are correctly initialized! (results/probe_direction.npy missing)')

print('Loading ESM-2...')
tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
esm2 = EsmForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D').to(DEVICE).eval()
print('Models loaded.')

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Device: cuda


Loading ESM-2...


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 6356.26it/s]


Models loaded.


In [5]:
def embed_esm2(sequences, batch_size=16):
    """Utility to embed sequences using ESM-2 layer 33"""
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(sequences), batch_size):
            batch_seqs = [s[:510] for s in sequences[i:i+batch_size]] # Truncate to avoid length errors
            inputs = tokenizer(batch_seqs, return_tensors='pt', padding=True, truncation=True).to(DEVICE)
            outputs = esm2(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states[BEST_LAYER] # (B, L, D)
            
            # Mean pooling over valid sequence lengths (ignoring padding)
            mask = inputs['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            all_embs.append(pooled.cpu().numpy())
            
            if (i + batch_size) % 1000 == 0:
                print(f'Embedded {i + batch_size}/{len(sequences)}')
                
    return np.concatenate(all_embs, axis=0)

def predict_proba(embs):
    """Get probability scores from embeddings"""
    with torch.no_grad():
        t_embs = torch.tensor(embs, dtype=torch.float32).to(DEVICE)
        logits = probe(t_embs)
        return torch.sigmoid(logits).cpu().numpy()

## Dataset 1: UniRef50 (Negative / Control Sequences)
Estimating False Positive Rate (FPR) on a large, diverse set of natural non-toxic sequences.

In [6]:
uniref_fpr = None
try:
    print('Loading UniRef50 from FASTA...')
    # Using standard SeqIO since evodiff.data.UniRefDataset is not exposed in this version
    from Bio import SeqIO
    import os, random
    uniref_path = 'data/uniref50.fasta'  # Ensure you place the fasta here
    if os.path.exists(uniref_path):
        control_sequences = [str(r.seq) for r in SeqIO.parse(uniref_path, 'fasta')]
    else:
        raise FileNotFoundError(f'{uniref_path} not found. Please download UniRef50 to data/')
    
    # Subsample 10,000 for tractability
    random.seed(42)
    controls = random.sample(control_sequences, min(10000, len(control_sequences)))
    print(f'Sampled {len(controls)} UniRef50 sequences.')
    
    control_embeddings = embed_esm2(controls)
    control_scores = predict_proba(control_embeddings)
    uniref_fpr = (control_scores > 0.5).mean()
    print(f'\nUniRef50 False Positive Rate: {uniref_fpr:.3%}')
except Exception as e:
    print(f'UniRef50 test skipped: {e}')


Loading UniRef50 from FASTA...
Sampled 10000 UniRef50 sequences.
Embedded 2000/10000
Embedded 4000/10000
Embedded 6000/10000
Embedded 8000/10000
Embedded 10000/10000

UniRef50 False Positive Rate: 46.420%


## Dataset 2: OpenFold MSAs (Structure-Conditioned Hard Negatives)
Sequences sharing the same fold as toxins, but performing non-toxic functions. This tests specificity under worst-case conditions.

In [7]:
hard_neg_fpr = None
try:
    # Example structure for loading hard negatives. 
    # In practice, you would filter MSAs by TM-score > 0.5 against toxin structures.
    print('Loading valid MSAs...')
    valid_msas = pd.read_csv('data/valid_msas.csv')
    
    # Placeholder for actual Foldseek / TM-score logic
    hard_negatives = [] 
    if len(hard_negatives) > 0:
        hard_neg_embs = embed_esm2(hard_negatives)
        hard_neg_scores = predict_proba(hard_neg_embs)
        hard_neg_fpr = (hard_neg_scores > 0.5).mean()
        print(f'\nHard Negative False Positive Rate: {hard_neg_fpr:.3%}')
    else:
        print('No hard negatives generated. Requires TM-score filtering logic against OpenFold MSAs.')
except Exception as e:
    print(f'Hard negative test skipped: {e}')

Loading valid MSAs...
No hard negatives generated. Requires TM-score filtering logic against OpenFold MSAs.


## Dataset 3: EvoDiff Redesigns
Redesigning toxins purely in sequence space (ignoring structure) to test the structural encoding hypothesis.

In [8]:
evodiff_results = {}
try:
    from evodiff.pretrained import OA_DM_38M
    print('Loading EvoDiff OA_DM_38M model...')
    model, collater, evo_tokenizer, scheme = OA_DM_38M()
    model = model.to(DEVICE).eval()
    
    # Load original toxin sequences for starting points
    tox_recs = list(SeqIO.parse('data/toxins_clustered.fasta', 'fasta'))
    toxin_seqs = [str(r.seq) for r in tox_recs]
    
    print(f'Generating unconditional redesigns for {min(100, len(toxin_seqs))} toxins...')
    redesigns = []
    
    # Unconditional generation via masking and inpainting
    # NOTE: EvoDiff's generate API typically requires specific tensor inputs and masks.
    # This is a conceptual loop based on the roadmap.
    # 
    # for toxin_seq in toxin_seqs[:100]:
    #     for _ in range(10):
    #         # Pseudo-code for EvoDiff generation
    #         masked = mask_random_positions(toxin_seq, mask_frac=0.5)
    #         generated = model.generate(masked, n_steps=500)
    #         redesigns.append(generated)
    
    # Once generated, we evaluate:
    if len(redesigns) > 0:
        evodiff_embs = embed_esm2(redesigns)
        evodiff_scores = predict_proba(evodiff_embs)
        evodiff_detection = (evodiff_scores > 0.5).mean()
        print(f'\nEvoDiff Probe Detection Rate: {evodiff_detection:.3%}')
        evodiff_results['detection_rate'] = evodiff_detection
    else:
        print('EvoDiff generation not executed (placeholder logic).')
        
except Exception as e:
    print(f'EvoDiff redesign generation skipped: {e}')

Loading EvoDiff OA_DM_38M model...
Downloading: "https://zenodo.org/record/8045076/files/oaar-38M.tar?download=1" to /home/ubuntu/.cache/torch/hub/checkpoints/oaar-38M.tar


100%|██████████| 434M/434M [00:42<00:00, 10.7MB/s] 


Generating unconditional redesigns for 100 toxins...
EvoDiff generation not executed (placeholder logic).


In [9]:
import os, urllib.request
import pandas as pd

idr_path = 'data/idr_sequences.csv'
if not os.path.exists(idr_path):
    print('Downloading 1000 Intrinsically Disordered Regions (IDRs) from UniProt...')
    url = 'https://rest.uniprot.org/uniprotkb/stream?query=keyword:KW-0941&format=fasta'
    req = urllib.request.Request(url)
    count = 0
    idr_seqs = []
    current_seq = ""
    with urllib.request.urlopen(req) as response:
        for line in response:
            line_str = line.decode('utf-8').strip()
            if line_str.startswith('>'):
                if current_seq:
                    idr_seqs.append(current_seq)
                    current_seq = ""
                    count += 1
                if count >= 1000:
                    break
            else:
                current_seq += line_str
    pd.DataFrame({'sequence': idr_seqs}).to_csv(idr_path, index=False)
    print(f'Successfully downloaded {len(idr_seqs)} IDR sequences to {idr_path}')
else:
    print(f'{idr_path} already exists.')


Successfully downloaded 1000 IDR sequences to data/idr_sequences.csv


## Dataset 4: IDR Data (Reverse Homology)
Intrinsically disordered regions — structureless negative controls.

In [10]:
idr_fpr = None
try:
    print('Loading IDR sequences...')
    idr_df = pd.read_csv('data/idr_sequences.csv')
    idr_seqs = idr_df['sequence'].tolist()[:1000] # Cap at 1000
    
    idr_embs = embed_esm2(idr_seqs)
    idr_scores = predict_proba(idr_embs)
    idr_fpr = (idr_scores > 0.5).mean()
    
    print(f'\nIDR mean probe score: {idr_scores.mean():.3f}')
    print(f'IDR False Positive Rate: {idr_fpr:.3%}')
except Exception as e:
    print(f'IDR test skipped: {e}')

Loading IDR sequences...

IDR mean probe score: 0.959
IDR False Positive Rate: 100.000%


## Summary Results Table

In [11]:
results_summary = {
    'UniRef50_FPR': uniref_fpr,
    'Hard_Negatives_FPR': hard_neg_fpr,
    'EvoDiff_Detection': evodiff_results.get('detection_rate', None),
    'IDR_FPR': idr_fpr
}

print('=== Out-of-Distribution Generalisation Summary ===')
print(f'UniRef50 Controls FPR:       {uniref_fpr if uniref_fpr is not None else "N/A":.1%}' if isinstance(uniref_fpr, float) else f'UniRef50 Controls FPR:       N/A')
print(f'OpenFold Hard Negs FPR:      {hard_neg_fpr if hard_neg_fpr is not None else "N/A":.1%}' if isinstance(hard_neg_fpr, float) else f'OpenFold Hard Negs FPR:      N/A')
print(f'IDR Structureless FPR:       {idr_fpr if idr_fpr is not None else "N/A":.1%}' if isinstance(idr_fpr, float) else f'IDR Structureless FPR:       N/A')
print(f'EvoDiff Redesign Detection:  {results_summary["EvoDiff_Detection"] if results_summary["EvoDiff_Detection"] is not None else "N/A":.1%}' if isinstance(results_summary["EvoDiff_Detection"], float) else f'EvoDiff Redesign Detection:  N/A')

with open('results/evodiff/ood_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print('\nSaved summary to results/evodiff/ood_summary.json')

=== Out-of-Distribution Generalisation Summary ===
UniRef50 Controls FPR:       46.4%
OpenFold Hard Negs FPR:      N/A
IDR Structureless FPR:       100.0%
EvoDiff Redesign Detection:  N/A

Saved summary to results/evodiff/ood_summary.json
